# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > 练习实验：用于基于内容过滤的深度学习

在本练习中，你将使用神经网络实现基于内容的过滤，构建电影推荐系统。

# 大纲 <img align="left" src="./images/film_reel.png"     style=" width:40px;  " >
- [ 1 - 软件包](#1)
- [ 2 - 电影评分数据集](#2)
  - [ 2.1 使用神经网络进行基于内容的过滤](#2.1)
  - [ 2.2 准备训练数据](#2.2)
- [ 3 - 用于基于内容过滤的神经网络](#3)
  - [ 3.1 预测](#3.1)
    - [ 练习 1](#ex01)
- [ 4 - 恭喜！](#4)

<a name="1"></a>
## 1——软件包 <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的 NumPy、TensorFlow，以及 [scikit-learn](https://scikit-learn.org/stable/) 中的实用例程。还将使用 [tabulate](https://pypi.org/project/tabulate/) 整齐地打印表格，并使用 [Pandas](https://pandas.pydata.org/) 组织表格数据。

In [ ]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2——电影评分数据集 <img align="left" src="./images/film_rating.png" style=" width:40px;" >
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。

[F. Maxwell Harper 和 Joseph A. Konstan，2015，《MovieLens 数据集：历史与背景》，ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19。<https://doi.org/10.1145/2827872>]

原始数据集包含 600 位用户对 9,000 部电影的评分，评分范围为 0.5 到 5，步长为 0.5。为了重点研究 2000 年以来的电影和热门类型，数据集经过了精简。精简后的数据集包含 $n_u = 395$ 位用户和 $n_m= 694$ 部电影。对于每部电影，数据集提供片名、上映日期和一个或多个类型。例如，《玩具总动员 3》于 2010 年上映，包含多种类型：“冒险|动画|儿童|喜剧|奇幻|IMAX”。除了评分之外，该数据集几乎不包含用户信息。下面介绍的神经网络将使用此数据集创建训练向量。

<a name="2.1"></a>
### 2.1 使用神经网络进行基于内容的过滤

在协同过滤实验中，您生成了两个向量：用户向量和项目／电影向量，它们的点积用于预测评分。这些向量完全由评分推导而来。

基于内容的过滤同样会生成用户特征向量和电影特征向量，但它会考虑可能存在的其他用户和／或电影信息，这些信息可能改善预测。附加信息被输入神经网络，随后由网络生成如下所示的用户向量和电影向量。
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>
提供给网络的电影内容由原始数据和一些“工程化特征”组合而成。回顾课程 1 第 2 周实验 4 中关于特征工程的讨论和实验。原始特征包括电影发行年份，以及以独热向量表示的电影类型，共有 14 种类型。工程化特征是根据用户评分计算的平均评分。具有多种类型的电影，每种类型都会对应一个训练向量。

用户内容仅由工程化特征组成。针对每位用户，计算其在每种类型上的平均评分。此外，还有用户 ID、评分数量和平均评分等信息可用，但它们不包含在训练或预测内容中；这些信息有助于解释数据。

训练集由数据集中用户给出的全部评分组成。用户向量与电影／项目向量会一起作为训练集输入上述网络。对于同一用户评过的所有电影，其用户向量都相同。

下面加载并显示部分数据。

In [ ]:
# Load Data, set configuration variables
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
scaledata = True  # applies the standard scalar to data if true
print(f"Number of training vectors: {len(item_train)}")

部分用户和物品/电影特征不用于训练。下面，方括号“[]”中的特征（例如“用户 ID”“评分次数”和“平均评分”）在训练和使用模型时不会被纳入。请注意，对于该用户评过分的所有电影，用户向量都是相同的。

In [ ]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

In [ ]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

In [ ]:
print(f"y_train[:5]: {y_train[:5]}")

从上面可以看到，电影 6874 是一部 2003 年上映的动作片。用户 2 对动作片的平均评分为 3.9。此外，电影 6874 还被归入犯罪片和惊悚片类型。MovieLens 用户给该电影的平均评分为 4。一个训练样本由两个表中的各一行以及 y_train 中的一个评分组成。

<a name="2.2"></a>
### 2.2 准备训练数据
回顾课程 1 第 2 周的内容，您曾探索用特征缩放改善收敛。我们将使用 [scikit-learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 缩放输入特征。课程 1 第 2 周实验 5 中也使用过它。下面还展示了如何使用 inverse_transform 还原原始输入。

In [ ]:
# scale training data
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

为了能够评估结果，我们会按照课程 2 第 3 周中讨论的方式，将数据划分为训练集和测试集。这里，我们将使用 [sklearn train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来拆分并打乱数据。请注意，将初始随机状态设为相同的值，可以确保物品、用户和 y 以完全相同的方式被打乱。

In [ ]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

经过缩放并打乱顺序的数据现在均值为零。

In [ ]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

使用 Min-Max 缩放器对目标评分进行缩放，使目标值位于 -1 和 1 之间。我们使用 scikit-learn，是因为它提供 inverse_transform。[scikit-learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [ ]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

<a name="3"></a>
## 3 - 用于基于内容过滤的神经网络
现在，让我们按照上图的说明构建一个神经网络。它将包含两个通过点积组合的网络，这两个网络都由你来构建。在本例中，它们将完全相同。请注意，这两个网络并非必须相同。如果用户内容明显多于电影内容，你可以选择让用户网络比电影网络更复杂。在本例中，两类内容较为相似，因此两个网络保持相同。

- 使用 Keras 顺序模型
    - 第一层是包含 256 个单元并使用 ReLU 激活函数的密集层。
    - 第二层是包含 128 个单元并使用 ReLU 激活函数的密集层。
    - 第三层是包含 `num_outputs` 个单元且使用线性激活或不使用激活函数的密集层。
    
网络的其余部分将直接提供。所提供的代码没有使用 Keras 顺序模型，而是使用 Keras 的[函数式 API](https://keras.io/guides/functional_api/)。这种格式为组件之间的连接方式提供了更大的灵活性。

In [ ]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###   
      
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     

    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = Model([input_user, input_item], output)

model.summary()

In [ ]:
# Public tests
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  可以按如下方式创建带 ReLU 激活函数的密集层。
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看答案</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

    

我们将使用均方误差损失和 Adam 优化器。

In [ ]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [ ]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

评估模型，以确定测试数据上的损失。该损失与训练损失相近，说明模型没有明显过拟合训练数据。

In [ ]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

<a name="3.1"></a>
### 3.1 预测
下面，你将在多种情况下使用模型进行预测。
#### 为新用户进行预测
首先，我们将创建一个新用户，并让模型为该用户推荐电影。在示例用户内容上尝试此示例后，可以随意修改用户内容以匹配你自己的偏好，然后查看模型的推荐。请注意，评分范围为 0.5 到 5.0（含），增量为 0.5。

In [ ]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])


让我们看看新用户评分最高的电影。回顾一下，该用户向量所表示的类型偏好是喜剧片和爱情片。
下面，我们将使用一组电影/物品向量 `item_vecs`，其中训练集/测试集中的每部电影都有一个对应向量。将其与上面的用户向量进行匹配，并使用缩放后的向量，为上面的新用户预测所有电影的评分。

In [ ]:
# generate and replicate the user vector to match the number movies in the data set.
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

如果您确实在上面创建了一个用户，请注意：该网络经过训练，会根据包含一**组**用户类型评分的用户向量来预测用户评分。如果训练数据中没有类型评分组合相似的用户，那么仅为某一种类型提供最高评分、而为其余类型提供最低评分，对网络来说可能没有意义。

#### 对现有用户的预测
让我们看看对数据集中一位用户“user 36”的预测。可以将预测评分与模型中的评分进行比较。请注意，具有多个类型的电影会在训练数据中出现多次。例如，“The Time Machine”包含三个类型：冒险、动作、科幻。

In [ ]:
uid =  36 
# form a set of user vectors. This is the same vector, transformed and repeated.
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

#print sorted predictions
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

#### 查找相似物品
上面的神经网络会生成两个特征向量：用户特征向量 $v_u$ 和电影特征向量 $v_m$。它们都是包含 32 个条目的向量，其值难以解释。不过，相似物品将具有相似的向量。可以利用这些信息进行推荐。例如，如果某位用户给“Toy Story 3”打了高分，就可以通过选择具有相似电影特征向量的电影来推荐相似影片。

一种相似度度量是两个向量 $ \mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的距离平方：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex01"></a>
### 练习 1

编写一个计算平方距离的函数。

In [ ]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###     
    
    ### END CODE HERE ###     
    return (d)

In [ ]:
# Public tests
test_sq_dist(sq_dist)

In [ ]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  求和通常意味着应使用 for 循环，但这里可以用一条语句执行逐元素减法。此外，可以使用 np.square 对减法结果逐元素取平方，再使用 np.sum 对平方后的元素求和。
    
</details>

    

电影之间的距离矩阵可以在模型训练完成时计算一次，然后无需重新训练，便可复用于新的推荐。模型训练完成后的第一步，是获得每部电影的电影特征向量 $v_m$。为此，我们将使用训练好的 `item_NN` 构建一个小模型，让电影向量通过它来生成 $v_m$。

In [ ]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # incorporate normalization as was done in the original model
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

有了电影模型后，可以使用该模型，以一组物品/电影向量作为输入进行预测，从而创建一组电影特征向量。`item_vecs` 是包含所有电影向量的集合。请记住，同一部电影会针对其每种类型分别出现为一个向量。必须对它进行缩放，才能与训练好的模型配合使用。预测结果是每部电影对应的一个 32 项特征向量。

In [ ]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

现在计算每个电影特征向量与所有其他电影特征向量之间的距离平方矩阵：
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

然后，我们可以通过查找每一行的最小值来找到最相近的电影。我们将使用 [NumPy 掩码数组](https://numpy.org/doc/1.21/user/tutorial-ma.html) 来避免选择同一部电影。计算时不会包含对角线上的掩码值。

In [ ]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

结果表明，模型会推荐相同类型的电影。

<a name="4"></a>
## 4 - 恭喜！<img align="left" src="./images/film_award.png" style=" width:40px;">
你已经完成了一个基于内容的推荐系统。

这种结构是许多商业推荐系统的基础。如果有更多用户信息可用，可以大幅扩展用户内容。推荐项目也不限于电影：它可用于推荐任何项目，例如书籍、汽车，或与你“购物车”中某个项目相似的项目。